In [1]:
import os
from glob import glob
import pandas as pd
from functools import reduce
from xml.etree import ElementTree as et

In [6]:
#load xml files
xml_list = glob('./*.xml')
xml_list = list(map(lambda x: x.replace('\\','/'),xml_list))

In [7]:
xml_list

['./00236.xml',
 './00249.xml',
 './00386.xml',
 './00136.xml',
 './00145.xml',
 './00215.xml',
 './00490.xml',
 './00223.xml',
 './00289.xml',
 './00017.xml',
 './00400.xml',
 './00134.xml',
 './00451.xml',
 './00328.xml',
 './00397.xml',
 './00094.xml',
 './00443.xml',
 './00433.xml',
 './00206.xml',
 './00452.xml',
 './00533.xml',
 './00272.xml',
 './00118.xml',
 './00210.xml',
 './00174.xml',
 './00155.xml',
 './00405.xml',
 './00237.xml',
 './00133.xml',
 './00166.xml',
 './00564.xml',
 './00412.xml',
 './00407.xml',
 './00280.xml',
 './00234.xml',
 './00323.xml',
 './00169.xml',
 './00358.xml',
 './00086.xml',
 './00028.xml',
 './00442.xml',
 './00187.xml',
 './00090.xml',
 './00362.xml',
 './00253.xml',
 './00322.xml',
 './00113.xml',
 './00359.xml',
 './00345.xml',
 './00262.xml',
 './00461.xml',
 './00325.xml',
 './00163.xml',
 './00221.xml',
 './00366.xml',
 './00536.xml',
 './00509.xml',
 './00472.xml',
 './00432.xml',
 './00268.xml',
 './00127.xml',
 './00422.xml',
 './0000

In [8]:
# read xml files and extract filename, size(width,height), object(name,xmin,xmax,ymin,ymax)

def extract_text(filename):
    tree = et.parse(filename)
    root = tree.getroot()
    image_name = root.find('filename').text
    width = root.find('size').find('width').text
    height = root.find('size').find('height').text

    objs = root.findall('object')
    parser = []
    for obj in objs:
        name = obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = bndbox.find('xmin').text
        xmax = bndbox.find('xmax').text
        ymin = bndbox.find('ymin').text
        ymax = bndbox.find('ymax').text
        parser.append([image_name, width, height, name, xmin, xmax, ymin, ymax])

    return parser

parser_all = list(map(extract_text,xml_list))
data = reduce(lambda x,y: x+y, parser_all)

In [9]:
df = pd.DataFrame(data,columns = ['filename','width','height','name','xmin','xmax','ymin','ymax'])
df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax
0,00236.jpg,600,450,car,20,573,137,421
1,00249.jpg,1280,853,car,630,1221,317,579
2,00386.jpg,1280,1024,car,67,1230,320,826
3,00136.jpg,1280,853,car,304,379,658,700
4,00136.jpg,1280,853,car,975,1185,654,738


In [10]:
df.shape

(815, 8)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815 entries, 0 to 814
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   filename  815 non-null    object
 1   width     815 non-null    object
 2   height    815 non-null    object
 3   name      815 non-null    object
 4   xmin      815 non-null    object
 5   xmax      815 non-null    object
 6   ymin      815 non-null    object
 7   ymax      815 non-null    object
dtypes: object(8)
memory usage: 51.1+ KB


In [12]:
# type conversion
cols = ['width', 'height', 'xmin', 'xmax', 'ymin', 'ymax']
df[cols] = df[cols].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815 entries, 0 to 814
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   filename  815 non-null    object
 1   width     815 non-null    int64 
 2   height    815 non-null    int64 
 3   name      815 non-null    object
 4   xmin      815 non-null    int64 
 5   xmax      815 non-null    int64 
 6   ymin      815 non-null    int64 
 7   ymax      815 non-null    int64 
dtypes: int64(6), object(2)
memory usage: 51.1+ KB


In [13]:
# center x and center y
df['center_x'] = ((df['xmax'] + df['xmin'])/2)/df['width']
df['center_y'] = ((df['ymax'] + df['ymin'])/2)/df['height']

# w: width of the bounding box
df['w'] = (df['xmax'] - df['xmin'])/df['width']

# h" height of the bounding box
df['h'] = (df['ymax'] - df['ymin'])/df['height']

In [14]:
df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h
0,00236.jpg,600,450,car,20,573,137,421,0.494167,0.620000,0.921667,0.631111
1,00249.jpg,1280,853,car,630,1221,317,579,0.723047,0.525205,0.461719,0.307151
2,00386.jpg,1280,1024,car,67,1230,320,826,0.506641,0.559570,0.908594,0.494141
3,00136.jpg,1280,853,car,304,379,658,700,0.266797,0.796014,0.058594,0.049238
4,00136.jpg,1280,853,car,975,1185,654,738,0.843750,0.815944,0.164062,0.098476


In [17]:
# 80% of images for training the model
images = df['filename'].unique()
img_df = pd.DataFrame(images,columns=['filename'])
img_train = tuple(img_df.sample(frac=0.8, random_state=42)['filename'])
img_test = tuple(img_df[~img_df['filename'].isin(img_train)]['filename'])
img_train

('00188.jpg',
 '0436.jpg',
 '00490.jpg',
 '00205.jpg',
 '00290.jpg',
 '00534.jpg',
 '00459.jpg',
 '00260.jpg',
 '00213.jpg',
 '00265.jpg',
 '00111.jpg',
 '00388.jpg',
 '00068.jpg',
 '00556.jpg',
 '00400.jpg',
 '00423.jpg',
 '00104.jpg',
 '00564.jpg',
 '00392.jpg',
 '00390.jpg',
 '00537.jpg',
 '00027.jpg',
 '00481.jpg',
 '00408.jpg',
 '00154.jpg',
 '00045.jpg',
 '00263.jpg',
 '00176.jpg',
 '00458.jpg',
 '00469.jpg',
 '00266.jpg',
 '00235.jpg',
 '00218.jpg',
 '00017.jpg',
 '00088.jpg',
 '00200.jpg',
 '00343.jpg',
 '00084.jpg',
 '00135.jpg',
 '00146.jpg',
 '00332.jpg',
 '00297.jpg',
 '00536.jpg',
 '00196.jpg',
 '00599.jpg',
 '00162.jpg',
 '00098.jpg',
 '00149.jpg',
 '00379.jpg',
 '00329.jpg',
 '00453.jpg',
 '00463.jpg',
 '00246.jpg',
 '00082.jpg',
 '00524.jpg',
 '00416.jpg',
 '00300.jpg',
 '00320.jpg',
 '00386.jpg',
 '00106.jpg',
 '00231.jpg',
 '00411.jpg',
 '00378.jpg',
 '00022.jpg',
 '00363.jpg',
 '00020.jpg',
 '00498.jpg',
 '00203.jpg',
 '00439.jpg',
 '00280.jpg',
 '00394.jpg',
 '00361

In [18]:
len(img_train), len(img_test)

(417, 104)

In [19]:
train_df = df.query(f'filename in {img_train}')
test_df = df.query(f'filename in {img_test}')

In [20]:
train_df.head()


,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h
0,00236.jpg,600,450,car,20,573,137,421,0.494167,0.620000,0.921667,0.631111
2,00386.jpg,1280,1024,car,67,1230,320,826,0.506641,0.559570,0.908594,0.494141
3,00136.jpg,1280,853,car,304,379,658,700,0.266797,0.796014,0.058594,0.049238
4,00136.jpg,1280,853,car,975,1185,654,738,0.843750,0.815944,0.164062,0.098476
5,00136.jpg,1280,853,car,620,813,663,762,0.559766,0.835287,0.150781,0.116061


In [21]:
test_df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h
1,00249.jpg,1280,853,car,630,1221,317,579,0.723047,0.525205,0.461719,0.307151
19,00328.jpg,1280,960,car,80,689,376,746,0.300391,0.584375,0.475781,0.385417
30,00533.jpg,500,312,car,42,483,68,261,0.525000,0.527244,0.882000,0.618590
31,00272.jpg,1280,960,car,363,534,544,617,0.350391,0.604688,0.133594,0.076042
32,00272.jpg,1280,960,car,460,640,673,751,0.429688,0.741667,0.140625,0.081250


In [22]:
# label encoding
def label_encoding(x):
    labels = {'person':0, 'car':1, 'chair':2, 'bottle':3, 'dog':4, 'bicycle':5, 'motorbike':6}
    return labels[x]

train_df['id'] = train_df['name'].apply(label_encoding)
test_df['id'] = test_df['name'].apply(label_encoding)

/tmp/ipykernel_2123/3604471879.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['id'] = train_df['name'].apply(label_encoding)
/tmp/ipykernel_2123/3604471879.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['id'] = test_df['name'].apply(label_encoding)


In [23]:
train_df.head(10)

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h,id
0,00236.jpg,600,450,car,20,573,137,421,0.494167,0.620000,0.921667,0.631111,1
2,00386.jpg,1280,1024,car,67,1230,320,826,0.506641,0.559570,0.908594,0.494141,1
3,00136.jpg,1280,853,car,304,379,658,700,0.266797,0.796014,0.058594,0.049238,1
4,00136.jpg,1280,853,car,975,1185,654,738,0.843750,0.815944,0.164062,0.098476,1
5,00136.jpg,1280,853,car,620,813,663,762,0.559766,0.835287,0.150781,0.116061,1
6,00136.jpg,1280,853,car,790,966,666,738,0.685937,0.822978,0.137500,0.084408,1
7,00145.jpg,1280,853,car,235,808,311,590,0.407422,0.528136,0.447656,0.327081,1
8,00145.jpg,1280,853,car,749,1280,335,570,0.792578,0.530481,0.414844,0.275498,1
9,00215.jpg,1280,853,car,1,1126,166,618,0.440234,0.459555,0.878906,0.529894,1
10,00490.jpg,1280,848,car,143,1166,283,574,0.511328,0.505307,0.799219,0.343160,1


In [24]:
test_df.head(10)

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h,id
1,00249.jpg,1280,853,car,630,1221,317,579,0.723047,0.525205,0.461719,0.307151,1
19,00328.jpg,1280,960,car,80,689,376,746,0.300391,0.584375,0.475781,0.385417,1
30,00533.jpg,500,312,car,42,483,68,261,0.525000,0.527244,0.882000,0.618590,1
31,00272.jpg,1280,960,car,363,534,544,617,0.350391,0.604688,0.133594,0.076042,1
32,00272.jpg,1280,960,car,460,640,673,751,0.429688,0.741667,0.140625,0.081250,1
33,00272.jpg,1280,960,car,480,665,439,517,0.447266,0.497917,0.144531,0.081250,1
34,00272.jpg,1280,960,car,870,1043,618,689,0.747266,0.680729,0.135156,0.073958,1
35,00272.jpg,1280,960,car,702,878,343,417,0.617188,0.395833,0.137500,0.077083,1
36,00272.jpg,1280,960,car,667,852,528,607,0.593359,0.591146,0.144531,0.082292,1
37,00272.jpg,1280,960,car,940,1121,415,488,0.805078,0.470313,0.141406,0.076042,1


In [28]:
# Create train and test folder
import os
from shutil import move

train_folder = 'Dataset/train'
test_folder = 'Dataset/test'

os.makedirs(train_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

In [29]:
cols = ['filename','id','center_x','center_y','w','h']
groupby_obj_train = train_df[cols].groupby('filename')
groupby_obj_test = test_df[cols].groupby('filename')

In [39]:
# Save images and labels in respective train and test folder in text format
def save_data(filename,folder_path,group_obj):
    # Define source and destination paths for the image
    src = os.path.join('.',filename)
    dst = os.path.join(folder_path,filename)

    # Check if the source image file exists before moving
    if os.path.exists(src):
        move(src,dst)

        # Save the labels if the image was successfully moved
        text_filename = os.path.join(folder_path,
                                     os.path.splitext(filename)[0]+'.txt')
        group_obj.get_group(filename).set_index('filename').to_csv(text_filename,sep=' ',index=False,header=False)
    else:
        print(f"Warning: Image file not found for {filename}. Skipping this entry.")

In [40]:
filename_series = pd.Series(groupby_obj_train.groups.keys())

In [41]:
filename_series.apply(save_data,args=(train_folder,groupby_obj_train))

,0
0,None
1,None
2,None
3,None
4,None
...,...
412,None
413,None
414,None
415,None


In [42]:
filename_series_test = pd.Series(groupby_obj_test.groups.keys())
filename_series_test.apply(save_data,args=(test_folder,groupby_obj_test))

,0
0,None
1,None
2,None
3,None
4,None
...,...
99,None
100,None
101,None
102,None
